In [ ]:
# Part (a)
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

data_path = Path('OJ.csv')
if not data_path.exists():
    raise FileNotFoundError('OJ.csv was not found in the current directory.')

oj_df = pd.read_csv(data_path)
oj_df = oj_df.drop(columns=['STORE', 'Store7'])

train_df, test_df = train_test_split(
    oj_df,
    train_size=800,
    random_state=RANDOM_STATE,
    stratify=oj_df['Purchase']
)

print(f'Training set size: {len(train_df)} observations')
print(f'Test set size: {len(test_df)} observations')
train_df.head()

In [ ]:
# Part (b)
from sklearn.tree import DecisionTreeClassifier

feature_cols = [col for col in train_df.columns if col != 'Purchase']
X_train = train_df[feature_cols]
y_train = train_df['Purchase']

unpruned_tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
unpruned_tree.fit(X_train, y_train)

train_predictions = unpruned_tree.predict(X_train)
train_accuracy = (train_predictions == y_train).mean()
num_terminal_nodes = unpruned_tree.get_n_leaves()

print('Decision tree model:
', unpruned_tree)
print('
Training set metrics:')
print(f'Training accuracy: {train_accuracy:.4f}')
print(f'Training error rate: {1 - train_accuracy:.4f}')
print(f'Number of terminal nodes: {num_terminal_nodes}')

feature_importance = (
    pd.Series(unpruned_tree.feature_importances_, index=feature_cols)
    .sort_values(ascending=False)
)
display(feature_importance.to_frame('importance'))

In [ ]:
# Part (c)
from sklearn.tree import export_text

text_representation = export_text(unpruned_tree, feature_names=feature_cols)
print(text_representation)

import numpy as np
terminal_nodes = np.where(unpruned_tree.tree_.children_left == -1)[0]
first_leaf = int(terminal_nodes[0])
node_samples = int(unpruned_tree.tree_.n_node_samples[first_leaf])
node_class_counts = unpruned_tree.tree_.value[first_leaf][0]
predicted_class = unpruned_tree.classes_[np.argmax(node_class_counts)]
print('
First terminal node details:')
print(f'Node index: {first_leaf}')
print(f'Number of samples: {node_samples}')
print(f'Class counts: {dict(zip(unpruned_tree.classes_, node_class_counts.astype(int)))}')
print(f'Predicted class (yval): {predicted_class}')

In [ ]:
# Part (d)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import plot_tree

plt.figure(figsize=(22, 12))
plot_tree(
    unpruned_tree,
    feature_names=feature_cols,
    class_names=unpruned_tree.classes_,
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title('Unpruned Decision Tree for Orange Juice Purchase')
plt.tight_layout()
plt.savefig('unpruned_decision_tree.png', dpi=300)
plt.show()

In [ ]:
# Part (e)
from sklearn.metrics import confusion_matrix, accuracy_score

X_test = test_df[feature_cols]
y_test = test_df['Purchase']

test_predictions = unpruned_tree.predict(X_test)
cm = confusion_matrix(y_test, test_predictions, labels=unpruned_tree.classes_)
cm_df = pd.DataFrame(
    cm,
    index=[f'Actual {cls}' for cls in unpruned_tree.classes_],
    columns=[f'Predicted {cls}' for cls in unpruned_tree.classes_]
)

print('Confusion matrix (test set):')
display(cm_df)

test_accuracy = accuracy_score(y_test, test_predictions)
test_error_rate = 1 - test_accuracy
print(f'
Test accuracy: {test_accuracy:.4f}')
print(f'Test error rate: {test_error_rate:.4f}')

plt.figure(figsize=(6, 5))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix on Test Data')
plt.tight_layout()
plt.savefig('test_confusion_matrix.png', dpi=300)
plt.show()

In [ ]:
# Part (f)
from sklearn.model_selection import StratifiedKFold, cross_val_score

ccp_path = unpruned_tree.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = ccp_path.ccp_alphas[:-1]

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

cv_records = []

for alpha in ccp_alphas:
    clf = DecisionTreeClassifier(random_state=RANDOM_STATE, ccp_alpha=alpha)
    cv_scores = cross_val_score(clf, X_train, y_train, cv=skf, scoring='accuracy')
    clf.fit(X_train, y_train)
    cv_records.append(
        {
            'alpha': alpha,
            'tree_size': clf.get_n_leaves(),
            'cv_accuracy_mean': cv_scores.mean(),
            'cv_accuracy_std': cv_scores.std()
        }
    )

cv_results = pd.DataFrame(cv_records)

In [ ]:
# Part (g)
if cv_results.empty:
    raise ValueError('Cross-validation results are empty. Please run the previous cell first.')

cv_results = cv_results.sort_values('tree_size').reset_index(drop=True)
cv_results['cv_error'] = 1 - cv_results['cv_accuracy_mean']

plt.figure(figsize=(8, 5))
sns.lineplot(data=cv_results, x='tree_size', y='cv_error', marker='o')
plt.xlabel('Number of Terminal Nodes')
plt.ylabel('Cross-Validated Classification Error Rate')
plt.title('Cross-Validated Error vs. Tree Size')
plt.tight_layout()
plt.savefig('cv_error_vs_tree_size.png', dpi=300)
plt.show()
cv_results.head()

In [ ]:
# Part (h)
best_row = cv_results.loc[cv_results['cv_error'].idxmin()].copy()
best_tree_size = int(best_row['tree_size'])
best_alpha = float(best_row['alpha'])

print(f'Lowest cross-validated classification error rate: {best_row["cv_error"]:.4f}')
print(f'Optimal tree size (number of terminal nodes): {best_tree_size}')
print(f'Associated complexity parameter (alpha): {best_alpha:.6f}')

In [ ]:
# Part (i)
desired_tree_size = best_tree_size
desired_alpha = best_alpha

if desired_tree_size == unpruned_tree.get_n_leaves():
    desired_tree_size = 5
    candidate_alphas = cv_results.loc[cv_results['tree_size'] == desired_tree_size, 'alpha']
    if not candidate_alphas.empty:
        desired_alpha = float(candidate_alphas.iloc[0])
    else:
        closest_idx = (cv_results['tree_size'] - desired_tree_size).abs().idxmin()
        desired_alpha = float(cv_results.loc[closest_idx, 'alpha'])

pruned_tree = DecisionTreeClassifier(random_state=RANDOM_STATE, ccp_alpha=desired_alpha)
pruned_tree.fit(X_train, y_train)

print(f'Pruned tree alpha: {desired_alpha:.6f}')
print(f'Pruned tree size (number of terminal nodes): {pruned_tree.get_n_leaves()}')

In [ ]:
# Part (j)
def compute_error_rates(model, X_tr, y_tr, X_te, y_te):
    train_pred = model.predict(X_tr)
    test_pred = model.predict(X_te)
    train_acc = accuracy_score(y_tr, train_pred)
    test_acc = accuracy_score(y_te, test_pred)
    return {
        'training_error': 1 - train_acc,
        'test_error': 1 - test_acc
    }

unpruned_errors = compute_error_rates(unpruned_tree, X_train, y_train, X_test, y_test)
pruned_errors = compute_error_rates(pruned_tree, X_train, y_train, X_test, y_test)

comparison_df = pd.DataFrame(
    [unpruned_errors, pruned_errors],
    index=['Unpruned tree', 'Pruned tree']
)

display(comparison_df)

In [ ]:
# Part II (a)

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

print("Standardizing features for SVM models...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Feature scaling completed.")

print("Training linear SVM with cost=0.01...")
linear_svm_c001 = SVC(kernel='linear', C=0.01, random_state=RANDOM_STATE)
linear_svm_c001.fit(X_train_scaled, y_train)
print("Linear SVM training finished.")

print('Linear SVM (cost=0.01) summary:')
print(f'Kernel: {linear_svm_c001.kernel}')
print(f'Cost (C): {linear_svm_c001.C}')
print(f'Number of features: {linear_svm_c001.n_features_in_}')
print(f'Number of support vectors: {linear_svm_c001.n_support_.sum()}')

support_vector_df = pd.DataFrame({
    'class': linear_svm_c001.classes_,
    'support_vectors': linear_svm_c001.n_support_
})
display(support_vector_df)

coef_series = pd.Series(linear_svm_c001.coef_.ravel(), index=feature_cols)
coef_summary = coef_series.to_frame('coefficient').sort_values(
    by='coefficient', key=lambda s: s.abs(), ascending=False
)
display(coef_summary)


In [ ]:
# Part II (b)

linear_c001_errors = compute_error_rates(linear_svm_c001, X_train_scaled, y_train, X_test_scaled, y_test)
print('Linear SVM (cost=0.01) error rates:')
for name, value in linear_c001_errors.items():
    pretty_name = name.replace('_', ' ').title()
    print(f'{pretty_name}: {value:.4f}')


In [ ]:
# Part II (c)

from sklearn.model_selection import GridSearchCV, StratifiedKFold

linear_cost_values = [0.01, 0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10]
linear_param_grid = {'C': linear_cost_values}
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

print(f'Starting linear SVM cost tuning over {len(linear_cost_values)} values with 3-fold CV...')
linear_grid = GridSearchCV(
    estimator=SVC(kernel='linear', random_state=RANDOM_STATE),
    param_grid=linear_param_grid,
    scoring='accuracy',
    cv=cv_strategy,
    n_jobs=-1,
    return_train_score=False,
    verbose=1
)
linear_grid.fit(X_train_scaled, y_train)
print('Completed linear SVM cost tuning.')

linear_cv_results = (
    pd.DataFrame(linear_grid.cv_results_)[['param_C', 'mean_test_score', 'std_test_score', 'rank_test_score']]
    .rename(columns={'param_C': 'C', 'mean_test_score': 'mean_accuracy', 'std_test_score': 'std_accuracy'})
    .sort_values('C')
    .reset_index(drop=True)
)
print('Cross-validated accuracy for linear SVM:')
display(linear_cv_results)

best_linear_cost = float(linear_grid.best_params_['C'])
print(f'Optimal cost based on cross-validation: {best_linear_cost}')


In [ ]:
# Part II (d)

print(f'Training linear SVM with optimal cost C={best_linear_cost}...')
best_linear_svm = SVC(kernel='linear', C=best_linear_cost, random_state=RANDOM_STATE)
best_linear_svm.fit(X_train_scaled, y_train)
print('Training completed.')

best_linear_errors = compute_error_rates(best_linear_svm, X_train_scaled, y_train, X_test_scaled, y_test)
print(f'Linear SVM with optimal cost (C={best_linear_cost}) error rates:')
for name, value in best_linear_errors.items():
    pretty_name = name.replace('_', ' ').title()
    print(f'{pretty_name}: {value:.4f}')


In [ ]:
# Part II (e)

print('Training radial SVM with cost=0.01...')
radial_svm_c001 = SVC(kernel='rbf', C=0.01, gamma='scale', random_state=RANDOM_STATE)
radial_svm_c001.fit(X_train_scaled, y_train)
print('Radial SVM training finished.')

radial_initial_errors = compute_error_rates(radial_svm_c001, X_train_scaled, y_train, X_test_scaled, y_test)
print('Radial SVM (cost=0.01) error rates:')
for name, value in radial_initial_errors.items():
    pretty_name = name.replace('_', ' ').title()
    print(f'{pretty_name}: {value:.4f}')

radial_cost_values = [0.01, 0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10]
radial_param_grid = {'C': radial_cost_values}

print(f'Starting radial SVM cost tuning over {len(radial_cost_values)} values with 3-fold CV...')
radial_grid = GridSearchCV(
    estimator=SVC(kernel='rbf', gamma='scale', random_state=RANDOM_STATE),
    param_grid=radial_param_grid,
    scoring='accuracy',
    cv=cv_strategy,
    n_jobs=-1,
    return_train_score=False,
    verbose=1
)
radial_grid.fit(X_train_scaled, y_train)
print('Completed radial SVM cost tuning.')

radial_cv_results = (
    pd.DataFrame(radial_grid.cv_results_)[['param_C', 'mean_test_score', 'std_test_score', 'rank_test_score']]
    .rename(columns={'param_C': 'C', 'mean_test_score': 'mean_accuracy', 'std_test_score': 'std_accuracy'})
    .sort_values('C')
    .reset_index(drop=True)
)
print('Cross-validated accuracy for radial SVM:')
display(radial_cv_results)

best_radial_cost = float(radial_grid.best_params_['C'])
print(f'Optimal cost based on cross-validation: {best_radial_cost}')

print(f'Training radial SVM with optimal cost C={best_radial_cost}...')
best_radial_svm = SVC(kernel='rbf', C=best_radial_cost, gamma='scale', random_state=RANDOM_STATE)
best_radial_svm.fit(X_train_scaled, y_train)
print('Training completed.')

radial_best_errors = compute_error_rates(best_radial_svm, X_train_scaled, y_train, X_test_scaled, y_test)
print(f'Radial SVM with optimal cost (C={best_radial_cost}) error rates:')
for name, value in radial_best_errors.items():
    pretty_name = name.replace('_', ' ').title()
    print(f'{pretty_name}: {value:.4f}')


In [ ]:
# Part II (f)

print('Training polynomial (degree=2) SVM with cost=0.01...')
poly_svm_c001 = SVC(kernel='poly', degree=2, C=0.01, gamma='scale', coef0=0.0, random_state=RANDOM_STATE)
poly_svm_c001.fit(X_train_scaled, y_train)
print('Polynomial SVM training finished.')

poly_initial_errors = compute_error_rates(poly_svm_c001, X_train_scaled, y_train, X_test_scaled, y_test)
print('Polynomial SVM (degree=2, cost=0.01) error rates:')
for name, value in poly_initial_errors.items():
    pretty_name = name.replace('_', ' ').title()
    print(f'{pretty_name}: {value:.4f}')

poly_cost_values = [0.01, 0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10]
poly_param_grid = {'C': poly_cost_values}

print(f'Starting polynomial SVM cost tuning over {len(poly_cost_values)} values with 3-fold CV...')
poly_grid = GridSearchCV(
    estimator=SVC(kernel='poly', degree=2, gamma='scale', coef0=0.0, random_state=RANDOM_STATE),
    param_grid=poly_param_grid,
    scoring='accuracy',
    cv=cv_strategy,
    n_jobs=-1,
    return_train_score=False,
    verbose=1
)
poly_grid.fit(X_train_scaled, y_train)
print('Completed polynomial SVM cost tuning.')

poly_cv_results = (
    pd.DataFrame(poly_grid.cv_results_)[['param_C', 'mean_test_score', 'std_test_score', 'rank_test_score']]
    .rename(columns={'param_C': 'C', 'mean_test_score': 'mean_accuracy', 'std_test_score': 'std_accuracy'})
    .sort_values('C')
    .reset_index(drop=True)
)
print('Cross-validated accuracy for polynomial SVM:')
display(poly_cv_results)

best_poly_cost = float(poly_grid.best_params_['C'])
print(f'Optimal cost based on cross-validation: {best_poly_cost}')

print(f'Training polynomial SVM with optimal cost C={best_poly_cost}...')
best_poly_svm = SVC(kernel='poly', degree=2, C=best_poly_cost, gamma='scale', coef0=0.0, random_state=RANDOM_STATE)
best_poly_svm.fit(X_train_scaled, y_train)
print('Training completed.')

poly_best_errors = compute_error_rates(best_poly_svm, X_train_scaled, y_train, X_test_scaled, y_test)
print(f'Polynomial SVM with optimal cost (C={best_poly_cost}) error rates:')
for name, value in poly_best_errors.items():
    pretty_name = name.replace('_', ' ').title()
    print(f'{pretty_name}: {value:.4f}')


In [ ]:
# Part II (g)

model_comparisons = pd.DataFrame({
    'Unpruned tree': unpruned_errors,
    'Pruned tree': pruned_errors,
    'Linear SVM (C=0.01)': linear_c001_errors,
    'Linear SVM (CV best)': best_linear_errors,
    'RBF SVM (C=0.01)': radial_initial_errors,
    'RBF SVM (CV best)': radial_best_errors,
    'Poly SVM deg2 (C=0.01)': poly_initial_errors,
    'Poly SVM deg2 (CV best)': poly_best_errors
}).T
model_comparisons = model_comparisons.rename(columns={'training_error': 'Training error', 'test_error': 'Test error'})
display(model_comparisons)

best_model_name = model_comparisons['Test error'].idxmin()
best_test_error = model_comparisons['Test error'].min()
print(f'Lowest test error is achieved by: {best_model_name} (test error = {best_test_error:.4f})')
